# DX702 Coding Quiz: Week 12

## Imports

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm # For linear regression
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from scipy.stats import t,skew
from tqdm import tqdm
from scipy.stats import skew
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression


In [2]:
plt.rcParams['axes.titlesize']  = 10
plt.rcParams['axes.labelsize']  = 8
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['lines.markersize'] = 3
plt.rcParams['axes.edgecolor']  = 'gray'
plt.rcParams['xtick.color']     = 'gray'
plt.rcParams['ytick.color'] = 'gray'
plt.rcParams['xtick.color'] = 'gray'
plt.rcParams['ytick.color'] = 'gray'
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['xtick.labelsize'] = 8

## Functions

In [3]:
data = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/refs/heads/main/homework_12.1.csv')
data = data.drop(columns=['Unnamed: 0'])
data

,Y,Time,Group
0,-0.232900,-1.193204,0
1,2.848846,-1.607748,1
2,0.550209,-0.269793,0
3,2.198280,7.743730,0
4,4.111044,-4.244359,1
...,...,...,...
9995,5.188977,2.930515,0
9996,3.557366,0.275730,0
9997,2.208786,8.377715,0
9998,2.593101,-4.406393,0


## Question 1  
**10 Points**  

Find the effect in dataset 12.1, given that Group = 1 is the treatment group and Time > 0 is the treatment time. 

Option A
1

Option B
2

Option C
0.5

Option D
1.5



In [ ]:
# binary indicator for post-treatment period
data["Post"]        = (data["Time"] > 0).astype(int)

# interaction term for DiD
data["Treatment"]   = data["Group"] * data["Post"]

# Run DiD regression
model               = smf.ols("Y ~ Group + Post + Treatment", data = data).fit()

# Extract treatment effect (coefficient of the interaction term)
treatment_effect    = model.params["Treatment"]

treatment_effect


0.944010378352991

## Question 2  
**10 Points**  

With dataset 12.2, run a test to check prior trends. To do this, run a linear regression on the data before Time = 0, with an interaction term equal to Group x Time. Which is the closest to the t-value of the interaction term? 

Option A: 25

***Option B: 5***

Option C: 0.02

Option D: 1




In [5]:
data_2 = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/refs/heads/main/homework_12.2.csv')
data_2 = data_2.drop(columns=['Unnamed: 0'])

In [ ]:
# Filter data for Time < 0
df_pre_treatment = data_2[data_2['Time'] < 0]

# Run linear regression with interaction term Group x Time; 
# the formula specifies Y as dependent variable, and Group, Time, and their interaction as independent variables
model = smf.ols('Y ~ Group + Time + Group:Time', data = df_pre_treatment).fit()

# t-stat of interaction term
t_stat_interaction = model.tvalues['Group:Time']

print(f"t-stat of interaction term (Group x Time): {t_stat_interaction}")


t-stat of interaction term (Group x Time): 5.329379663923893


## Question 3  
**10 Points**  

With the following, make an error such that the correlation (`np.corrcoef`) between the error at `time t and time t + 1` is 0.80 on average, the mean error is 0, and the standard deviation of the error is 1. (These may be approximate.) There are 10,000 items. Given: 

`X = this error `

`Y = 2 * X + this error` (recalculated with new randomness relative to the previous line) 



The standard error of the estimated X coefficient computed via Ordinary Least Squares (OLS) basic standard error should be about 0.01. Then the standard error computed via simulation (the standard deviation of the estimated coefficient over many trials) is closest to: 



Note: you might consider Googling / asking ChatGPT how to generate an autocorrelated time series with a specific correlation and standard deviation. 

***Option A: 0.02***

Option B : 0.0025

Option C: 0.005

Option D: 0.01

<font color='plum'>The simulation-based estimate is significantly larger than the standard error reported by OLS. This is expected because:

OLS assumes independent errors, and when this assumption is violated (e.g., with correlated errors like a random walk), the standard errors it reports are underestimated.
The simulation captures the true variability of the coefficient under the correlated error structure.

## Question 4  
**10 Points**  

Find the variance inflation factor of X1: 

    np.random.seed(0) 
    X1 = np.random.normal(0, 1, 1000) 
    X2 = np.random.normal(0, 1, 1000) + X1 
    X3 = np.random.normal(0, 1, 1000) + 2 * X2 

Option A: 8

Option B: 16

Option C: 4

***Option D: 2***

In [ ]:
# Generate  variables
n_samples   = 10000
X1          = np.random.normal(0, 1, n_samples)
X2          = np.random.normal(0, 1, n_samples) + X1
X3          = np.random.normal(0, 1, n_samples) + 2 * X2

# To find VIF for X1, regress X1 on X2 and X3
# Prepare independent variables (X2 and X3) for  regression
# Add constant to independent variables for the intercept
X_regress_X1 = sm.add_constant(np.column_stack((X2, X3)))

# regression
model_X1    = sm.OLS(X1, X_regress_X1)
results_X1  = model_X1.fit()

# Get R-squared
r_squared_X1 = results_X1.rsquared

# VIF for X1
vif_X1 = 1 / (1 - r_squared_X1)

print(f"R-squared for regression of X1 on X2 and X3: {r_squared_X1:.4f}")
print(f"Variance Inflation Factor (VIF) for X1: {vif_X1:.4f}")

R-squared for regression of X1 on X2 and X3: 0.5060
Variance Inflation Factor (VIF) for X1: 2.0242
